In [6]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

In [7]:
# ============================================
# LOAD CIFAR-10 DATASET (TRAINING)
# ============================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("CIFAR Train Shape:", x_train.shape)
print("CIFAR Test Shape :", x_test.shape)

CIFAR Train Shape: (50000, 32, 32, 3)
CIFAR Test Shape : (10000, 32, 32, 3)


In [8]:
# ============================================
# NORMALIZE CIFAR DATA
# ============================================

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = y_train.flatten()
y_test = y_test.flatten()

In [9]:
# ============================================
# LOAD CINIC-10 TEST DATASET
# ============================================

cinic_test = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/mengcius/cinic10/test',
    image_size=(32,32),
    batch_size=64,
    shuffle=False
)

Found 90000 files belonging to 10 classes.


I0000 00:00:1778345864.168081      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778345864.174348      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [10]:
# ============================================
# NORMALIZE CINIC DATA
# ============================================

cinic_test = cinic_test.map(
    lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)
)

In [11]:
# ============================================
# DATA AUGMENTATION
# ============================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [12]:
# ============================================
# INCEPTION MODULE
# ============================================

def inception_module(x, filters):

    branch1 = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        activation='relu'
    )(x)

    branch2 = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        activation='relu'
    )(x)

    branch2 = layers.Conv2D(
        filters,
        (3,3),
        padding='same',
        activation='relu'
    )(branch2)

    branch3 = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        activation='relu'
    )(x)

    branch3 = layers.Conv2D(
        filters,
        (5,5),
        padding='same',
        activation='relu'
    )(branch3)

    branch4 = layers.MaxPooling2D(
        (3,3),
        strides=1,
        padding='same'
    )(x)

    branch4 = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        activation='relu'
    )(branch4)

    output = layers.Concatenate()([
        branch1,
        branch2,
        branch3,
        branch4
    ])

    return output

In [13]:
# ============================================
# BUILD INCEPTION MODEL
# ============================================

def build_inception(input_shape=(32,32,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)

    x = layers.Conv2D(
        64,
        (3,3),
        padding='same',
        activation='relu'
    )(x)

    x = layers.MaxPooling2D((2,2))(x)

    x = inception_module(x, 32)

    x = inception_module(x, 64)

    x = layers.MaxPooling2D((2,2))(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation='softmax'
    )(x)

    model = models.Model(inputs, outputs)

    return model

In [14]:
# ============================================
# CREATE MODEL
# ============================================

model = build_inception()

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32, 32, 3) │          0 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │      1,792 │ sequential[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ max_pooling2d[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │      9,248 │ conv2d_2[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │     25,632 │ conv2d_4[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d_1[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ conv2d_1[0][0],   │
│ (Concatenate)       │ 128)              │            │ conv2d_3[0][0],   │
│                     │                   │            │ conv2d_5[0][0],   │
│                     │                   │            │ conv2d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 16, 16,    │      8,256 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 16, 16,    │      8,256 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ concatenate[0][0] │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 16, 16,    │      8,256 │ concatenate[0][0] │
│                     │ 64)               │            │                 

 Total params: 251,594 (982.79 KB)

 Trainable params: 251,594 (982.79 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# ============================================
# COMPILE MODEL
# ============================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [16]:
# ============================================
# TRAIN ON CIFAR-10
# ============================================

history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=40,
    validation_data=(x_test, y_test)
)

Epoch 1/40


I0000 00:00:1778345908.112450     198 cuda_dnn.cc:529] Loaded cuDNN version 91002


782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 18ms/step - accuracy: 0.2119 - loss: 2.0515 - val_accuracy: 0.3873 - val_loss: 1.6653
Epoch 2/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.4003 - loss: 1.6206 - val_accuracy: 0.4827 - val_loss: 1.3972
Epoch 3/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.4590 - loss: 1.4753 - val_accuracy: 0.4854 - val_loss: 1.4006
Epoch 4/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.4981 - loss: 1.3851 - val_accuracy: 0.5172 - val_loss: 1.3816
Epoch 5/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.5227 - loss: 1.3131 - val_accuracy: 0.5564 - val_loss: 1.2056
Epoch 6/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.5526 - loss: 1.2436 - val_accuracy: 0.5800 - val_loss: 1.1495
Epoch 7/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.5644 - loss: 1.2067 - val_accuracy: 0.6090 - val_loss: 1.0719
Epoch 8/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.5879 - loss: 1.1564 - val_accurac

In [17]:
# ============================================
# TEST ON CINIC-10
# ============================================

test_loss, test_acc = model.evaluate(cinic_test)

print("CIFAR → CINIC Inception Accuracy:", test_acc)

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 209s 148ms/step - accuracy: 0.6398 - loss: 1.1543
CIFAR → CINIC Inception Accuracy: 0.5908777713775635


In [18]:
# ============================================
# SAVE MODEL
# ============================================

model.save('/kaggle/working/cifar_to_cinic_inception.keras')